# H8 · Market outcomes: do stock returns follow the same pattern as accounting profits?

> **Hypothesis H8:** Listed companies in more developed countries earn lower real stock returns, as they do lower ROA (H0).

| | |
|---|---|
| **Why we ask** | ROA is an accounting number, slow and shaped by accounting rules. Share prices are forward-looking and react to country conditions within months. If the development–ROA link (H0) reflects something real about profitability, investors should price it. Return and risk are also what an investor in each country would care about. |
| **Prediction** | **Negative** association between the development index and the real calendar-year stock return (return net of national inflation). |
| **Data** | Monthly adjusted close prices from Yahoo Finance (2018–2025, local currency) for the 2,590 Yahoo companies, 2019–2024 calendar years (national inflation ends in 2024). Real return = (1 + December-to-December return) / (1 + national inflation) − 1, winsorised at 1% / 99%. Development index of the **previous** year. Size = rank of market capitalisation within country. |
| **Primary test (fixed before looking at the result)** | `real return (pp) ~ development index + sector + size rank + calendar year`, errors clustered by country, wild-bootstrap p-value. |
| **Verdict rule** | *Supported* if the effect has the predicted sign, the wild-bootstrap p < 0.05, and the sign holds in at least 80% of the robustness samples. *Partly supported* if it has the predicted sign and p < 0.10 (or p < 0.05 but robustness is below 80%). *Contradicted* if significantly the opposite way. Otherwise *not supported*. Returns are in local currency and are not adjusted for exchange rates; survivor bias is possible (see limits). |

**Honest framing.** Several hypotheses in this series (H1 to H3) were suggested by an exploratory scan of the same data, so their p-values are optimistic and a pass is a lead, not a confirmation. Where possible the result is **replicated on ESEF**, an independent source (which still shares some companies with Yahoo). All hypotheses are corrected for multiple testing together in `1.0_hypotheses.ipynb`. Everything is **association, not causation**.

In [1]:
import sys, pathlib, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src" / "analysis"))
import prep, utils
from utils import CONTROLS, fit_cluster, wild_cluster, spearman, boot_ci, partial_rho, bh, scan

con = prep.connect()
q = lambda sql: con.sql(sql).df()
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120, "display.float_format", "{:,.3f}".format)
px.defaults.template = "plotly_white"

country = prep.load_country(con)
dim, profile, profile_all, panel = country["dim"], country["profile"], country["profile_all"], country["panel"]
feat_cols = country["feat_cols"]
fy = prep.load_firms(con, country)               # Yahoo company-years (primary sample)
names = dim.name
MIN_FIRMS = prep.MIN_FIRMS
print(f"{len(dim)} countries | {len(feat_cols)} country features | {len(fy):,} Yahoo company-years of {fy.company_id.nunique():,} companies")

SAMPLES = {
 "all firms": lambda d: d,
 "excluding financials & real estate": lambda d: d[~d.sector.isin(["Financial Services", "Real Estate"])],
 "excluding small-sample countries": lambda d: d[~d.iso3.isin(dim.index[dim.small_firm_sample])],
 "excluding USA, China, Japan": lambda d: d[~d.iso3.isin(["USA", "CHN", "JPN"])],
 "fiscal years 2023 and later": lambda d: d[d.fiscal_year >= 2023],
 "EU countries only": lambda d: d[d.iso3.isin(dim.index[dim.is_eu])],
 "non-EU countries only": lambda d: d[~d.iso3.isin(dim.index[dim.is_eu])],
}
def robustness(focal, outcome, expected_sign, B=499, extra="", controls=CONTROLS, data=None):
    """Re-estimate the primary test in each sample. Returns a table and the share of samples with the expected sign."""
    rows = []
    for name, flt in SAMPLES.items():
        r = wild_cluster(focal, flt(fy if data is None else data), outcome=outcome, extra=extra, B=B, controls=controls)
        rows.append({"sample": name, "coef": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"],
                     "countries": r["clusters"], "firm-years": r["n"]})
    t = pd.DataFrame(rows).set_index("sample")
    same = (np.sign(t.coef) == expected_sign) if expected_sign else pd.Series(True, index=t.index)
    return t.assign(**{"expected sign": same}), float(same.mean())

34 countries | 35 country features | 10,220 Yahoo company-years of 2,590 companies


## 1 · Building the return panel
One row per company and calendar year. Annual return = last price of the year over last price of the previous year; volatility = standard deviation of monthly returns inside the year, annualised. Companies are kept only if both December prices exist.

In [2]:
px_ = pd.read_parquet(ROOT / "data" / "flat" / "prices_monthly.parquet")
px_["year"], px_["m"] = px_.month.dt.year, px_.month.dt.month
px_ = px_.sort_values(["ticker", "month"])
px_["mret"] = px_.groupby("ticker").adj_close.pct_change()
px_.loc[px_.groupby("ticker").month.diff().dt.days > 40, "mret"] = np.nan      # a missing month breaks the monthly return
dec = px_[px_.m == 12].set_index(["ticker", "year"]).adj_close
ret = (dec / dec.groupby(level=0).shift(1) - 1).rename("ret").reset_index()
ret = ret.merge(px_.groupby(["ticker", "year"]).mret.agg(vol=lambda s: s.std() * np.sqrt(12) if s.notna().sum() >= 10 else np.nan), on=["ticker", "year"], how="left")
ret = ret[ret.year.between(2019, 2024)].dropna(subset=["ret"])

comp = q("SELECT company_id, source_key AS ticker, iso3, sector, market_cap FROM dim_company WHERE source_id = 'YF'")
comp["sector"] = comp.sector.fillna("Unknown")
comp["size_rank"] = comp.groupby("iso3").market_cap.rank(pct=True)
ret = ret.merge(comp, on="ticker")
lag = prep.country_year_lagged(country)[["iso3", "year", "dev_z", "inst_z", "gdp_growth_z", "trade_gdp_z"]]
infl = panel[["iso3", "year", "inflation"]]
ret = ret.merge(lag, on=["iso3", "year"], how="left").merge(infl, on=["iso3", "year"], how="left")
ret["real_ret"] = ((1 + ret.ret) / (1 + ret.inflation / 100) - 1)
for c in ["ret", "real_ret"]:
    lo, hi = ret[c].quantile(prep.WINSOR); ret[c + "_w"] = ret[c].clip(lo, hi)
ret["real_ret_pp"] = ret.real_ret_w * 100; ret["ret_pp"] = ret.ret_w * 100; ret["vol_pp"] = ret.vol * 100
ret["fiscal_year"] = ret.year                       # so the shared robustness samples apply
print(f"{len(ret):,} company-years, {ret.company_id.nunique():,} companies, {ret.iso3.nunique()} countries, calendar years {ret.year.min()}–{ret.year.max()}")
print("Typical real return by year (median, pp):"); print(ret.groupby("year").real_ret_pp.median().round(1).to_string())

13,986 company-years, 2,470 companies, 34 countries, calendar years 2019–2024
Typical real return by year (median, pp):
year
2019    16.200
2020    -0.600
2021    14.000
2022   -13.900
2023     6.200
2024     3.300


## 2 · Market outcomes by country
Median real return and volatility of each country's listed companies against the development index. Countries need at least 10 companies.

In [3]:
cn = ret.groupby("iso3").agg(real_return=("real_ret_pp", "median"), volatility=("vol_pp", "median"), firms=("company_id", "nunique")).query("firms >= @MIN_FIRMS")
cn = cn.join(country["dev_idx"].rename("development index")).join(fy.groupby("iso3").roa_pp.median().rename("median ROA (pp)"))
fig = make_subplots(rows=1, cols=3, subplot_titles=[f"{c} (ρ = {spearman(cn['development index'], cn[c])[0]:+.2f})" for c in ["real_return", "volatility", "median ROA (pp)"]])
for k, c in enumerate(["real_return", "volatility", "median ROA (pp)"], 1):
    d = cn[["development index", c]].dropna()
    fig.add_trace(go.Scatter(x=d["development index"], y=d[c], mode="markers+text", text=d.index, textposition="top center", textfont_size=8, marker=dict(size=7, color="#4C78A8"), showlegend=False), row=1, col=k)
    b, a = np.polyfit(d["development index"], d[c], 1); xs = np.array([d["development index"].min(), d["development index"].max()])
    fig.add_trace(go.Scatter(x=xs, y=a + b * xs, mode="lines", line=dict(color="#E45756"), showlegend=False), row=1, col=k)
fig.update_xaxes(title_text="development index (SD)"); fig.update_layout(height=450, title="Median real return, volatility and ROA vs development, one point per country"); fig.show()
r1, p1, n1 = spearman(cn["real_return"], cn["median ROA (pp)"])
print(f"Do high-ROA countries also have high returns? ρ = {r1:+.2f} (p = {p1:.3f}, n = {n1})")

Do high-ROA countries also have high returns? ρ = +0.31 (p = 0.076, n = 33)


## 3 · Primary test (company level)

In [4]:
CTRL = "C(sector) + size_rank + C(year)"
primary = wild_cluster("dev_z", ret, outcome="real_ret_pp", controls=CTRL, B=1999)
print(f"Typical real return: {ret.real_ret_pp.median():.1f}% a year (median).")
print(f"+1 SD development → {primary['coef']:+.2f} pp real return (95% CI {primary['ci_low']:+.2f} to {primary['ci_high']:+.2f}).")
print(f"p (cluster-robust) = {primary['p_cluster']:.4f} | p (wild bootstrap) = {primary['p_wild']:.4f} | {primary['clusters']} countries, {primary['n']:,} company-years")

Typical real return: 4.0% a year (median).
+1 SD development → -0.79 pp real return (95% CI -3.36 to +1.78).
p (cluster-robust) = 0.5494 | p (wild bootstrap) = 0.5870 | 34 countries, 13,955 company-years


### Return, risk and valuation
Secondary outcomes for the same conditions: volatility (do richer countries have calmer markets?), downside (probability of losing more than 20% in a year) and valuation (market cap over book equity, latest year only, where price and accounts are in the same currency). Corrected together (BH).

In [5]:
ret["bigloss"] = (ret.ret < -0.20).astype(float) * 100
lat = fy.sort_values("fiscal_year").groupby("company_id").tail(1)[["company_id", "equity", "dev_z", "sector", "iso3", "size_rank", "fiscal_year"]]
lat = lat.merge(q("SELECT company_id, market_cap, currency, financial_currency FROM dim_company"), on="company_id")
lat = lat[(lat.equity > 0) & (lat.market_cap > 0) & (lat.financial_currency == lat.currency)].copy()
lat["log_pb"] = np.log(lat.market_cap / lat.equity); lo, hi = lat.log_pb.quantile([.01, .99]); lat["log_pb"] = lat.log_pb.clip(lo, hi)
sec = []
for lab, out, data, ctrl in [("real return (pp)  [primary]", "real_ret_pp", ret, CTRL), ("volatility (pp, annualised)", "vol_pp", ret.dropna(subset=["vol_pp"]), CTRL),
                              ("share of years losing >20% (pp)", "bigloss", ret, CTRL), ("log price / book (latest)", "log_pb", lat, "C(sector) + size_rank")]:
    r = wild_cluster("dev_z", data, outcome=out, controls=ctrl, B=999)
    sec.append({"outcome": lab, "coef per +1 SD": r["coef"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p (wild)": r["p_wild"], "n": r["n"]})
secondary = pd.DataFrame(sec).set_index("outcome"); secondary["q (BH)"] = bh(secondary["p (wild)"])
secondary

,coef per +1 SD,ci_low,ci_high,p (wild),n,q (BH)
outcome,,,,,,
real return (pp) [primary],-0.793,-3.363,1.777,0.591,13955,0.788
"volatility (pp, annualised)",-6.418,-11.877,-0.958,0.030,13954,0.120
share of years losing >20% (pp),-0.225,-2.126,1.676,0.841,13955,0.841
log price / book (latest),-0.108,-0.280,0.064,0.377,2214,0.754


### Year by year
The return gap between more and less developed countries for each calendar year (company-level model with sector and size controls, errors clustered by country). A pattern driven by one or two years (2020 crash, 2022 energy shock) would show up here.

In [6]:
rows = []
for y, g in ret.groupby("year"):
    m = fit_cluster("real_ret_pp ~ dev_z + C(sector) + size_rank", g.dropna(subset=["dev_z", "size_rank"]))
    rows.append({"year": y, "coef": m.params["dev_z"], "lo": m.params["dev_z"] - 1.96 * m.bse["dev_z"], "hi": m.params["dev_z"] + 1.96 * m.bse["dev_z"], "n": int(m.nobs)})
by_year = pd.DataFrame(rows).set_index("year")
fig = go.Figure(go.Scatter(x=by_year.index, y=by_year.coef, mode="lines+markers", error_y=dict(type="data", array=by_year.hi - by_year.coef, arrayminus=by_year.coef - by_year.lo)))
fig.add_hline(y=0, line_color="grey"); fig.update_layout(height=380, title="Development → real return (pp per +1 SD), by calendar year", xaxis_title="calendar year", yaxis_title="pp"); fig.show()
by_year.round(2)

,coef,lo,hi,n
year,,,,
2019,1.860,-3.810,7.530,2193
2020,1.730,-3.600,7.060,2225
2021,0.760,-3.680,5.200,2283
2022,-4.830,-7.890,-1.770,2372
2023,-3.920,-8.730,0.900,2422
2024,-0.350,-5.000,4.310,2460


## 4 · Robustness
The primary test in seven samples (wild bootstrap, 499 draws each).

In [7]:
rob, rob_share = robustness("dev_z", "real_ret_pp", -1, controls=CTRL, data=ret)
fig = px.scatter(rob.reset_index(), x="coef", y="sample", error_x=rob.ci_high - rob.coef, error_x_minus=rob.coef - rob.ci_low, title="Development → real stock return (pp per +1 SD) in different samples")
fig.add_vline(x=0, line_color="grey"); fig.update_layout(height=420, yaxis_autorange="reversed"); fig.show()
print(f"Predicted sign (negative) in {rob_share:.0%} of samples.")
rob

Predicted sign (negative) in 100% of samples.


,coef,ci_low,ci_high,p (wild),countries,firm-years,expected sign
sample,,,,,,,
all firms,-0.793,-3.363,1.777,0.584,34,13955,True
excluding financials & real estate,-0.970,-3.625,1.686,0.520,34,10977,True
excluding small-sample countries,-0.736,-3.323,1.851,0.620,29,13375,True
"excluding USA, China, Japan",-0.676,-3.305,1.954,0.648,31,12258,True
fiscal years 2023 and later,-2.178,-5.771,1.415,0.276,34,4882,True
EU countries only,-2.905,-6.499,0.689,0.192,21,6986,True
non-EU countries only,-0.101,-3.231,3.029,0.940,13,6969,True


## 5 · Replication
There are no prices for ESEF-only firms, so ESEF cannot be used. The check here is a **different time window of the same firms** (calendar years 2023–2024 only, after the pandemic and the 2022 shock), which is *not independent* of the main sample.

In [8]:
late = ret[ret.year >= 2023]
rep = wild_cluster("dev_z", late, outcome="real_ret_pp", controls="C(sector) + size_rank + C(year)", B=1999)
rep_same = bool(np.sign(rep["coef"]) == -1)
print(f"Years 2023–2024 ({rep['n']:,} company-years): {rep['coef']:+.2f} pp (95% CI {rep['ci_low']:+.2f} to {rep['ci_high']:+.2f}), wild p = {rep['p_wild']:.3f}")

Years 2023–2024 (4,882 company-years): -2.18 pp (95% CI -5.77 to +1.42), wild p = 0.275


## 6 · Verdict

In [9]:
v = utils.verdict(primary["coef"], primary["p_wild"], -1, rob_share, primary["clusters"])
vol = secondary.loc["volatility (pp, annualised)"]; pb = secondary.loc["log price / book (latest)"]
lines = [
 f"Primary: +1 SD development → {primary['coef']:+.2f} pp real stock return (typical {ret.real_ret_pp.median():.1f}%), wild p = {primary['p_wild']:.3f}, {primary['clusters']} countries, {primary['n']:,} company-years.",
 f"Robustness: predicted sign in {rob_share:.0%} of seven samples. Replication on 2023–2024 only (same firms, not independent): {rep['coef']:+.2f} pp (wild p = {rep['p_wild']:.3f}), " + ("same direction." if rep_same else "opposite direction."),
 f"Risk: +1 SD development → {vol['coef per +1 SD']:+.2f} pp annualised volatility (wild p = {vol['p (wild)']:.3f}); valuation: {pb['coef per +1 SD']:+.2f} in log price/book (wild p = {pb['p (wild)']:.3f}).",
 f"Country level, return vs ROA: ρ = {r1:+.2f} (n = {n1}).",
 f"VERDICT H8: {v.upper()}",
]
print("\n".join(lines))
utils.save_result("H8", id="H8", title="Stock returns and country development", statement="Listed companies in more developed countries earn lower real stock returns.",
                  expected_sign=-1, effect=primary["coef"], ci_low=primary["ci_low"], ci_high=primary["ci_high"], effect_unit="pp real stock return per +1 SD development",
                  p_primary=primary["p_wild"], p_method="wild cluster bootstrap", n=primary["n"], clusters=primary["clusters"], robust_share=rob_share,
                  replication_coef=rep["coef"], replication_p=rep["p_wild"], replication_same_sign=rep_same, verdict=v,
                  notebook="1.8_market_outcomes.ipynb", notes=lines)

Primary: +1 SD development → -0.79 pp real stock return (typical 4.0%), wild p = 0.587, 34 countries, 13,955 company-years.
Robustness: predicted sign in 100% of seven samples. Replication on 2023–2024 only (same firms, not independent): -2.18 pp (wild p = 0.275), same direction.
Risk: +1 SD development → -6.42 pp annualised volatility (wild p = 0.030); valuation: -0.11 in log price/book (wild p = 0.377).
Country level, return vs ROA: ρ = +0.31 (n = 33).
VERDICT H8: NOT SUPPORTED


### Limits
- **Local currency.** Returns are not converted to a common currency, and the inflation adjustment is the national consumer price index. A foreign investor's return would also include exchange-rate changes.
- **Survivor bias.** The company list is of firms listed *today* with price history; firms that were delisted or went bankrupt in 2019–2025 are missing, which flatters returns, especially where delistings are more common.
- **Few markets, shared shocks.** Returns within a country move together, so the effective sample is 34 countries, and a single year (2020 or 2022) can dominate.
- **Prices vs accounts.** Yahoo's monthly adjusted close can contain data errors for small markets.
- Association, not causation.

In [10]:
con.close()